# `07_bicycle_route_extent_metrics`: Bicycle route network extent metrics per spatial unit

## Introduction

### Purpose

This notebook computes extent metrics for the designated bicycle route network at three spatial levels: municipality, province, and H3 cell. For each level it produces three companion tables: per-unit metrics, descriptive statistics, and a length breakdown by OSM network hierarchy (icn / ncn / rcn / lcn / multiple). Per the pipeline diagram and thesis §3.4.6, this is the stage-07 extent-metric notebook for the designated branch; the parallel notebook `07_non_bicycle_route_extent_metrics` performs the same computation on the non-route scoping layer.

### What is being measured

This notebook examines *bicycle routes*, not the physical infrastructure that makes up those routes. In OpenStreetMap, bicycle routes are represented as route relations: collections of member ways that together form a continuous cycling itinerary. `bicycle_route_m_ways_distinct_classified` (consumed here via its stage-06 spatial join) is the set of distinct ways belonging to one or more such relations, with the tag-resolution columns from notebook 04.

A bicycle route may follow dedicated cycle tracks, painted bike lanes, low-traffic residential streets, rural roads, shared-use paths, or any combination thereof. The presence of a bicycle route is therefore not evidence of cycling infrastructure, nor an indicator of infrastructure quality, safety, comfort, or level of protection. What is measured is the extent and distribution of designated route networks as mapped in OpenStreetMap. The underlying signal is one of official designation: a way belongs to a bicycle route relation because a responsible authority (a municipality, province, or cycling organisation) included it in a named, signed cycling route. This says nothing about the physical conditions of the way itself. The companion infrastructure metrics (in `07_bicycle_route_infrastructure_per_side_metrics`) speak to physical provision; this notebook speaks only to designation. Thesis §3.3 and §3.5 make the same point: route membership is "an administrative and wayfinding decision, not a physical fact."

### Metrics

For each spatial unit, three metrics are computed:

- **`bicycle_route_length_km`**: total length of distinct ways belonging to one or more bicycle route relations within the unit. Larger units tend to have longer networks by virtue of covering more territory; total length alone is not sufficient for comparing units of different sizes.
- **`bicycle_route_density_km_per_km2`**: total route length per square kilometre of land area. Standardising by area enables comparison across spatial units of different sizes. High density does not imply uniform spatial distribution; routes may be concentrated in specific parts of a unit.
- **`bicycle_route_km_per_1000_capita`**: total route length relative to resident population, scaled to 1,000 inhabitants. The per-capita view the thesis uses alongside density (Table 3, "Per-capita metric").

`bicycle_route_length_km` is additionally broken down by network membership category (icn, ncn, rcn, lcn, multiple). Ways belonging to more than one network level are assigned `multiple`. Disaggregating by network category lets us distinguish, for example, a municipality with an extensive local cycling network from one whose similar overall route length is driven almost entirely by a single long-distance national or international route passing through it. Per thesis §3.3 (Table 4), the Dutch corpus is dominated by `rcn` (the node network, ~97% of relations).

### Inputs

- `bicycle_route_m_ways_distinct_classified_per_municipality`, `_per_province`, `_per_h3` from `06_bicycle_route_m_ways_distinct_classified_per_spatial_unit`, loaded from cache.
- `municipalities`, `provinces`, `h3_cells` from `03_boundaries_population` (for the left-join base so that every spatial unit appears in the output, including units with zero route coverage).

### Outputs

| Spatial unit | Metrics table | Descriptive stats | Length by network |
|---|---|---|---|
| Municipality | `bicycle_route_metrics_by_municipality` | `bicycle_route_stats_by_municipality` | `bicycle_route_length_by_network_by_municipality` |
| Province | `bicycle_route_metrics_by_province` | `bicycle_route_stats_by_province` | `bicycle_route_length_by_network_by_province` |
| H3 cell | `bicycle_route_metrics_by_h3` | `bicycle_route_stats_by_h3` | `bicycle_route_length_by_network_by_h3` |

### Dependencies on prior notebooks

- `06_bicycle_route_m_ways_distinct_classified_per_spatial_unit` (output read from cache rather than re-run).
- `03_boundaries_population.ipynb` (via `%run`): provides `municipalities`, `provinces`, `h3_cells`, plus the geopandas / lonboard imports the visualisation function inherits transitively.

### Downstream consumers

- The municipality-level metrics feed the headline extent reporting in thesis §"Results" and the urban–rural-gradient analysis.
- The province- and H3-level metrics feed the MAUP robustness check in stage 08 (thesis §3.4.6).
- Stage 09 *Synthesis* combines these extent metrics with the per-side infrastructure metrics for the UNECE matrix application.

### Table of contents

1. [Environment setup](#1-environment-setup)
2. [Core functions](#2-core-functions)
3. [Municipalities](#3-municipalities)
   - 3.1 [Compute metrics](#31-compute-metrics)
   - 3.2 [Descriptive statistics](#32-descriptive-statistics)
   - 3.3 [Visualisation](#33-visualisation)
   - 3.4 [Breakdown by network hierarchy](#34-breakdown-by-network-hierarchy)
4. [Provinces](#4-provinces)
   - 4.1 [Compute metrics](#41-compute-metrics)
   - 4.2 [Descriptive statistics](#42-descriptive-statistics)
   - 4.3 [Visualisation](#43-visualisation)
   - 4.4 [Breakdown by network hierarchy](#44-breakdown-by-network-hierarchy)
5. [H3 grid cells](#5-h3-grid-cells)
   - 5.1 [Compute metrics](#51-compute-metrics)
   - 5.2 [Descriptive statistics](#52-descriptive-statistics)
   - 5.3 [Visualisation](#53-visualisation)
   - 5.4 [Breakdown by network hierarchy](#54-breakdown-by-network-hierarchy)

---

## 1. Environment setup

### Libraries and extensions

In [1]:
from IPython.utils import io
from lonboard import PolygonLayer, Map
from lonboard.colormap import apply_continuous_cmap
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

### Loading shared variables

The three spatial-unit tables (`municipalities`, `provinces`, `h3_cells`), plus `geopandas`, `MaplibreBasemap`, and `CartoStyle`, are brought in via `%run` of `03_boundaries_population`. The three per-spatial-unit ways tables are loaded directly from the cached parquet outputs of `06_bicycle_route_m_ways_distinct_classified_per_spatial_unit`, which avoids re-running the stage-06 spatial join.

In [35]:
with io.capture_output() as captured:
    %run /home/vbo226/03_boundaries_population.ipynb

from pathlib import Path
import duckdb

Path("/local/data/vbo226/cache").mkdir(parents=True, exist_ok=True)

bicycle_route_m_ways_distinct_classified_per_municipality = duckdb.read_parquet(
    "/local/data/vbo226/cache/bicycle_route_m_ways_distinct_classified_per_municipality.parquet"
)

bicycle_route_m_ways_distinct_classified_per_province = duckdb.read_parquet(
    "/local/data/vbo226/cache/bicycle_route_m_ways_distinct_classified_per_province.parquet"
)

bicycle_route_m_ways_distinct_classified_per_h3_cell = duckdb.read_parquet(
    "/local/data/vbo226/cache/bicycle_route_m_ways_distinct_classified_per_h3.parquet"
)

In [24]:
# For each spatial unit (municipality, province and h3 cell), convert GeoDataFrame into Arrow for easy ingestion into DuckDB 
municipalities_arrow = municipalities.to_arrow()
provinces_arrow = provinces.to_arrow()
h3_cells_arrow = h3_cells.to_arrow()

---

## 2. Core functions

Three functions are defined in this section and applied across all three spatial-unit levels. `calculate_metrics_per_spatial_unit` and `visualisation_minmax_norm_interactive` are spatial-unit-agnostic and reused without modification in `07_non_bicycle_route_extent_metrics`. `pivot_by_network` is specific to this notebook: it handles the network-hierarchy breakdown that is meaningful only for designated bicycle route ways, where each way carries a network classification (`way_m_bicycle_network_type_class`) inherited from its parent relation. Both `calculate_metrics_per_spatial_unit` and `pivot_by_network` share the same parameterisation pattern: the spatial-unit table, the ways-per-spatial-unit table, and the relevant column names are passed explicitly, so the same function body handles municipalities, provinces, and H3 cells without modification.

### `calculate_metrics_per_spatial_unit`

Aggregates clipped way lengths by spatial unit, computes the three metrics, and left-joins to the complete spatial-unit table so that units with no bicycle route coverage appear in the output with zero-valued metrics rather than being silently dropped. This is the mechanism that keeps the four ferry-only Wadden municipalities (Schiermonnikoog, Terschelling, Ameland, Vlieland) visible in the output as zeros, consistent with thesis §3.3.

`NULLIF` guards against division by zero for area and population: if either is zero or null, the corresponding metric is returned as `NULL` rather than raising an error.

In [25]:
def calculate_metrics_per_spatial_unit(spatial_unit_table, ways_per_spatal_unit, spatial_unit_code, spatial_unit_geometry, spatial_unit_areakm2, spatial_unit_population, metric_prefix, spatial_unit_name=None):

    # Include spatial_unit_name in SELECT only if provided (not meaningful for H3 cells)
    name_select = f"s.{spatial_unit_name}," if spatial_unit_name is not None else ""    

    length_col = f"{metric_prefix}_length_km"
    density_col = f"{metric_prefix}_density_km_per_km2"
    per_capita_col = f"{metric_prefix}_km_per_1000_capita"

    return duckdb.sql(f"""
    WITH base AS (
        SELECT *, 
            clipped_length_meters / 1000 as clipped_length_km
        FROM {ways_per_spatal_unit}
    ),
    aggregation AS (
        SELECT 
            {spatial_unit_code}, 
            ROUND(SUM(clipped_length_km), 3) AS {length_col}
        FROM base
        GROUP BY {spatial_unit_code}
    )
    SELECT 
        s.{spatial_unit_code},
        {name_select}
        s.{spatial_unit_geometry},
        s.{spatial_unit_areakm2},
        s.{spatial_unit_population},
        COALESCE(a.bicycle_route_length_km, 0) AS {length_col},

        ROUND(COALESCE(a.bicycle_route_length_km, 0) / NULLIF(s.{spatial_unit_areakm2}, 0), 3) AS {density_col},
        ROUND(COALESCE(a.bicycle_route_length_km, 0) / NULLIF(s.{spatial_unit_population}, 0) * 1000, 3) AS {per_capita_col}

    FROM {spatial_unit_table} s
    LEFT JOIN aggregation a
        ON s.{spatial_unit_code} = a.{spatial_unit_code}
    """)

### `pivot_by_network`

Aggregates clipped way lengths by spatial unit and network-membership category (`way_m_bicycle_network_type_class`), pivots network categories into separate columns (icn / ncn / rcn / lcn / multiple), and left-joins to the complete spatial-unit table so that all units appear in the output. Units with no ways in a given network category receive a zero rather than `NULL`.

In [26]:
def pivot_by_network(spatial_unit_table, ways_per_spatial_unit, spatial_unit_code, spatial_unit_name=None):

    # Include spatial_unit_name in SELECT only if provided (not meaningful for H3 cells)
    name_select = f"s.{spatial_unit_name}," if spatial_unit_name is not None else ""
    
    # Aggregate clipped way lengths by spatial unit and network category,
    # pivot network categories into separate columns, and LEFT JOIN to the
    # spatial unit table to ensure all spatial units appear in the output,
    # including those with no ways (filled with 0).
    return duckdb.sql(f"""
    WITH aggregation AS (
        SELECT
            {spatial_unit_code},
            way_m_bicycle_network_type_class,
            ROUND(SUM(clipped_length_meters) / 1000, 3) AS length_km
        FROM {ways_per_spatial_unit}
        GROUP BY {spatial_unit_code}, way_m_bicycle_network_type_class
    ),
    pivoted AS (
        PIVOT aggregation
        ON way_m_bicycle_network_type_class
        USING FIRST(length_km)
        GROUP BY {spatial_unit_code}
    )
    SELECT
        s.{spatial_unit_code},
        {name_select}
        COALESCE(p.icn,      0) AS icn_length_km,
        COALESCE(p.ncn,      0) AS ncn_length_km,
        COALESCE(p.rcn,      0) AS rcn_length_km,
        COALESCE(p.lcn,      0) AS lcn_length_km,
        COALESCE(p.multiple, 0) AS multiple_length_km
    FROM {spatial_unit_table} s
    LEFT JOIN pivoted p ON s.{spatial_unit_code} = p.{spatial_unit_code}
    """)

### `visualization_minmax_norm_interactive`

Produces an interactive choropleth map for any metric column, normalising values to 0-1 via min–max scaling and a viridis colormap so that maps across metrics and spatial units are visually comparable. The function is reused without modification in `07_non_bicycle_route_extent_metrics`.

In [27]:
def visualization_minmax_norm_interactive(table, attribute):
    # Min-max normalization
    norm_values = (table[attribute] - table[attribute].min())/ (table[attribute].max() - table[attribute].min())

    # Define sequential cmap
    cmap = plt.colormaps['viridis']

    # Create polygon layer and visualize using a sequential cmap 
    layer = PolygonLayer.from_geopandas(table, get_fill_color=apply_continuous_cmap(norm_values, cmap), get_line_color='white', line_width_min_pixels=0.5, before_id='watername_ocean')

    # Basemap styling parameters
    basemap = MaplibreBasemap(mode='interleaved', style=CartoStyle.Positron)

    # Return map
    return Map(layer, basemap=basemap, view_state = {'longitude': 5.2913, 'latitude': 52.1326, 'zoom': 6})

---

---

## 3. Municipalities

### 3.1 Compute metrics

In [28]:
bicycle_route_metrics_by_municipality = calculate_metrics_per_spatial_unit(spatial_unit_table='municipalities_arrow', 
                                   ways_per_spatal_unit='bicycle_route_m_ways_distinct_classified_per_municipality', 
                                   metric_prefix='bicycle_route',
                                   spatial_unit_code='municipality_code', 
                                   spatial_unit_name='municipality_name', 
                                   spatial_unit_geometry='geometry', 
                                   spatial_unit_areakm2='area_km2', 
                                   spatial_unit_population='population')

# Convert to GeoDataFrame
bicycle_route_metrics_by_municipality_gdf = gpd.GeoDataFrame.from_arrow(bicycle_route_metrics_by_municipality.arrow())

### 3.2 Descriptive statistics

In [29]:
bicycle_route_stats_by_municipality = bicycle_route_metrics_by_municipality_gdf[['bicycle_route_length_km', 'bicycle_route_density_km_per_km2', 'bicycle_route_km_per_1000_capita']].describe()
bicycle_route_stats_by_municipality

,bicycle_route_length_km,bicycle_route_density_km_per_km2,bicycle_route_km_per_1000_capita
count,342.000000,342.000000,342.000000
mean,112.780257,1.169906,3.162751
std,88.624932,0.426549,2.630065
min,0.000000,0.000000,0.000000
25%,46.164000,0.939500,1.232500
50%,89.317000,1.133000,2.489000
75%,158.697500,1.363000,4.256750
max,502.794000,3.523000,14.801000


median and mean for bicycle_route_density_km_per_km2 (municipality) is almost the same, indicating that there is not much variation between municiplities in terms of bicycle route density. Interestly, there are some municipalities that have 0 bicycle route density indicating that no ways were designated as bicycle route. Those municipalities are Schiermonnikoog, Terschelling, Ameland, Vlieland, which are the island in the nord of the Netherlands. 

### 3.3 Visualisation


#### Interactive

In [30]:
map_metrics_by_municipality_bicycle_route_length_km = visualization_minmax_norm_interactive(bicycle_route_metrics_by_municipality_gdf, 'bicycle_route_length_km')
map_metrics_by_municipality_bicycle_route_density_km_per_km2 = visualization_minmax_norm_interactive(bicycle_route_metrics_by_municipality_gdf, 'bicycle_route_density_km_per_km2')
map_metrics_by_municipality_bicycle_route_km_per_1000_capita = visualization_minmax_norm_interactive(bicycle_route_metrics_by_municipality_gdf, 'bicycle_route_km_per_1000_capita')


map_ = widgets.HBox([
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Total bicycle route length (km)</h3>"),
        map_metrics_by_municipality_bicycle_route_length_km
    ], layout=widgets.Layout(flex="1")),
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Bicycle route density (km per km2)</h3>"),
        map_metrics_by_municipality_bicycle_route_density_km_per_km2
    ], layout=widgets.Layout(flex="1")),
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>bicycle_route_km_per_1000_capita</h3>"),
        map_metrics_by_municipality_bicycle_route_km_per_1000_capita
    ], layout=widgets.Layout(flex="1"))
])

display(map_)

#### Static

### 3.4 Breakdown by network hierarchy

In [33]:
bicycle_route_length_by_network_by_municipality = pivot_by_network('municipalities_arrow', 'bicycle_route_m_ways_distinct_per_municipality', 'municipality_code', 'municipality_name')

## 4. Provinces

### 4.1 Compute metrics

In [36]:
bicycle_route_metrics_by_province = calculate_metrics_per_spatial_unit(spatial_unit_table='provinces_arrow', 
                                   ways_per_spatal_unit='bicycle_route_m_ways_distinct_classified_per_province', 
                                   metric_prefix='bicycle_route',                                   
                                   spatial_unit_code='province_code', 
                                   spatial_unit_name='province_name', 
                                   spatial_unit_geometry='geometry', 
                                   spatial_unit_areakm2='area_km2', 
                                   spatial_unit_population='population')

# Convert to GeoDataFrame
bicycle_route_metrics_by_province_gdf = gpd.GeoDataFrame.from_arrow(bicycle_route_metrics_by_province.arrow())

### 4.2 Descriptive statistics

In [37]:
bicycle_route_stats_by_province = bicycle_route_metrics_by_province_gdf[['bicycle_route_length_km', 'bicycle_route_density_km_per_km2', 'bicycle_route_km_per_1000_capita']].describe()
bicycle_route_stats_by_province

,bicycle_route_length_km,bicycle_route_density_km_per_km2,bicycle_route_km_per_1000_capita
count,12.000000,12.000000,12.000000
mean,3214.237833,0.950917,3.000167
std,1389.360467,0.264267,1.502870
min,1458.896000,0.542000,1.003000
25%,2222.258750,0.722750,1.956500
50%,2890.986000,1.048500,2.923000
75%,3913.300000,1.167750,4.022000
max,5727.682000,1.258000,5.202000


### 4.3 Visualisation

#### Interactive 

In [38]:
map_metrics_by_province_bicycle_route_length_km = visualization_minmax_norm_interactive(bicycle_route_metrics_by_province_gdf, 'bicycle_route_length_km')
map_metrics_by_province_bicycle_route_density_km_per_km2 = visualization_minmax_norm_interactive(bicycle_route_metrics_by_province_gdf, 'bicycle_route_density_km_per_km2')
map_metrics_by_province_bicycle_route_km_per_1000_capita = visualization_minmax_norm_interactive(bicycle_route_metrics_by_province_gdf, 'bicycle_route_km_per_1000_capita')


map_ = widgets.HBox([
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Total bicycle route length (km)</h3>"),
        map_metrics_by_province_bicycle_route_length_km
    ], layout=widgets.Layout(flex="1")),
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Bicycle route density (km per km2)</h3>"),
        map_metrics_by_province_bicycle_route_density_km_per_km2
    ], layout=widgets.Layout(flex="1")),
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>bicycle_route_km_per_1000_capita</h3>"),
        map_metrics_by_province_bicycle_route_km_per_1000_capita
    ], layout=widgets.Layout(flex="1"))
])

display(map_)

### 4.4 Breakdown by network hierarchy

In [39]:
bicycle_route_length_by_network_by_muncipality = pivot_by_network('provinces_arrow', 'bicycle_route_m_ways_distinct_per_province', 'province_code', 'province_name')

## 5. H3 grid cells

### 5.1 Compute metrics

In [40]:
bicycle_route_metrics_by_h3_cell = calculate_metrics_per_spatial_unit(spatial_unit_table='h3_cells_arrow', 
                                   ways_per_spatal_unit='bicycle_route_m_ways_distinct_per_h3_cell', 
                                   metric_prefix='bicycle_route',                                                                         
                                   spatial_unit_code='h3_index', 
                                   spatial_unit_geometry='geometry', 
                                   spatial_unit_areakm2='area_km2', 
                                   spatial_unit_population='population')

# Convert to GeoDataFrame
bicycle_route_metrics_by_h3_cell_gdf = gpd.GeoDataFrame.from_arrow(bicycle_route_metrics_by_h3_cell.arrow())

### 5.2 Descriptive statistics

In [41]:
bicycle_route_stats_by_h3_cell = bicycle_route_metrics_by_h3_cell_gdf[['bicycle_route_length_km', 'bicycle_route_density_km_per_km2', 'bicycle_route_km_per_1000_capita']].describe()
bicycle_route_stats_by_h3_cell

,bicycle_route_length_km,bicycle_route_density_km_per_km2,bicycle_route_km_per_1000_capita
count,66530.000000,66530.000000,49675.000000
mean,0.578428,0.926966,22.379669
std,0.642198,1.028050,87.649819
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.464000,0.745000,3.147000
75%,0.962000,1.544000,13.381000
max,5.585000,8.814000,2341.000000


### 5.3 Visualisation

#### Interactive

In [42]:
map_metrics_by_h3_cell_bicycle_route_length_km = visualization_minmax_norm_interactive(bicycle_route_metrics_by_h3_cell_gdf, 'bicycle_route_length_km')
map_metrics_by_h3_cell_bicycle_route_density_km_per_km2 = visualization_minmax_norm_interactive(bicycle_route_metrics_by_h3_cell_gdf, 'bicycle_route_density_km_per_km2')
map_metrics_by_h3_cell_bicycle_route_km_per_1000_capita = visualization_minmax_norm_interactive(bicycle_route_metrics_by_h3_cell_gdf, 'bicycle_route_km_per_1000_capita')


map_ = widgets.HBox([
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Total bicycle route length (km)</h3>"),
        map_metrics_by_h3_cell_bicycle_route_length_km
    ], layout=widgets.Layout(flex="1")),
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Bicycle route density (km per km2)</h3>"),
        map_metrics_by_h3_cell_bicycle_route_density_km_per_km2
    ], layout=widgets.Layout(flex="1")),
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>bicycle_route_km_per_1000_capita</h3>"),
        map_metrics_by_h3_cell_bicycle_route_km_per_1000_capita
    ], layout=widgets.Layout(flex="1"))
])

display(map_)

/home/vbo226/.local/lib/python3.11/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(
/home/vbo226/.local/lib/python3.11/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(
/home/vbo226/.local/lib/python3.11/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(


### 5.4 Breakdown by network hierarchy

In [43]:
bicycle_route_length_by_network_by_h3_cell = pivot_by_network('h3_cells_arrow', 'bicycle_route_m_ways_distinct_per_h3_cell', 'h3_index')

### Export

Persist all nine output tables to disk so the downstream stage-08 MAUP notebook and stage-09 synthesis notebook can read them without re-running the metrics

In [30]:
from pathlib import Path

Path("/local/data/vbo226/cache").mkdir(parents=True, exist_ok=True)

outputs = {
    "bicycle_route_metrics_by_province": bicycle_route_metrics_by_province,
    "bicycle_route_stats_by_province": bicycle_route_stats_by_province,
    "bicycle_route_metrics_by_municipality": bicycle_route_metrics_by_municipality,
    "bicycle_route_stats_by_municipality": bicycle_route_stats_by_municipality,
    "bicycle_route_metrics_by_h3_cell": bicycle_route_metrics_by_h3_cell,
    "bicycle_route_stats_by_h3_cell": bicycle_route_stats_by_h3_cell,
    "bicycle_route_length_by_network_by_h3_cell": bicycle_route_length_by_network_by_h3_cell,
}

for name, df in outputs.items():
    df.to_parquet(f"/local/data/vbo226/cache/{name}.parquet")